In [ ]:
file_name = ""
tiled_id = ""

# pandaN.h5 Analysis Notebook

In [ ]:
# imports:
import numpy as np
import json
import matplotlib.pyplot as plt
import h5py
from scipy.signal import savgol_filter

In [ ]:
import os
print(os.listdir("/data"))

In [ ]:
# artifacts:
ARTIFACTS = {
    "notebook": "/tmp/notebook.html",
    "plot": "/tmp/plot_result.png",
    "pca_plot": "/tmp/plot_pca.png",
    "report": "/tmp/report.json",
    }

In [ ]:
# read dataset and get (x, y) data:
fpath = f"/data/{file_name}.nxs"

instrument_path = "/entry/instrument/panda1"
xname = "inenc-1-val_dataset"
yname = "fmc_in-val1_dataset"

meta_path = "/entry/plan_metadata/plan_args"

with h5py.File(fpath, "r") as f:

    # panda1 data
    x = f[f"{instrument_path}/{xname}"][...]
    y = f[f"{instrument_path}/{yname}"][...]

    # plan metadata
    npos = f[f"{meta_path}/capture_positions"].len()
    nrep = f[f"{meta_path}/num_seqtable_repeats"].asstr("utf-8")[()]
    ntraj = f[f"{meta_path}/num_trajectory_points"].asstr("utf-8")[()]
    nsweep = f[f"{meta_path}/number_of_sweeps"].asstr("utf-8")[()]
    nstep = f[f"{meta_path}/stepsize"].asstr("utf-8")[()]

In [ ]:
nrep = int(nrep); ntraj = int(ntraj); nsweep = int(nsweep); nstep = float(nstep)

ncol = int(y.shape[0]/nrep)

In [ ]:
x = x.reshape((nrep, ncol))

for i in range(ncol+1):
    start = i*nrep
    stop = (i+1)*(nrep)
    y[start:stop] = y[start:stop][::-1]


y = y.reshape((nrep, ncol))

## Utils for scan diagnostics:

In [ ]:
def get_scan_metrics(y:np.ndarray)->dict:

    win_length = int(len(y)/10)

    yf = savgol_filter(x=y, window_length=win_length, polyorder=2)
    noise = np.std(y-yf)

    res = {"min_y": np.min(y), "max_y": np.max(y),
            "noise": noise}
    return res


def repetition_processing(y:np.ndarray):
    """
    detect number of repetitions of scan.
    """
    print(f"Found {nrep} repetitions.")

    report = {}

    for i in range(nrep):

        res = get_scan_metrics(y[i,:])

        report[f"Repetition {i+1}"] = res

    cy = np.cumsum(y, axis=0)/np.sum(y, axis=0)

    noise = np.std(cy - savgol_filter(cy, int(len(y)/10), 2), axis=1)

    
    return report, noise

In [ ]:
report, noise = repetition_processing(y)

In [ ]:
report["snr"] = noise
report["x"] = x
report["y"] = y
report["tiled_id"] = tiled_id

with open(ARTIFACTS["report"], "w"):
    json.dumps(report, indent=4, default=str)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 5), layout="tight")
xplot = np.linspace(1, len(noise)+1, len(noise))

fig.suptitle(f"Data for: {fpath.split('/')[-1]}\ntiled id: {tiled_id}")

ax[0].set_title("xy data")
ax[0].plot(x.T, y.T, lw=1)

ax[1].set_title(r"$\frac{noise}{\sqrt{N}}$")
ax[1].plot(xplot, noise/np.sqrt(xplot), "o-");
ax[1].set_xlabel("Repetition No.")

[a.set_xlabel(xname) for a in [ax[0], ax[1]]]
[a.set_ylabel(yname) for a in [ax[0], ax[1]]]

fig.savefig(ARTIFACTS["plot"])
plt.show();

In [ ]:
from sklearn.decomposition import PCA

In [ ]:
ncomp = 10
model = PCA(n_components=ncomp)

In [ ]:
W = model.fit_transform(y.T)
H = model.components_

In [ ]:
fig, ax = plt.subplots(ncomp, 2, layout="tight", figsize=(10, 12))
for i in range(ncomp):
    ax[i, 0].set_title(f"PC{i+1} Scores")
    ax[i, 0].plot(np.arange(1, H.shape[1]+1), H[i,:], "x-")
    ax[i, 1].plot(x[i,:],W[:,i], "x-")
fig.savefig(ARTIFACTS["pca_plot"])
plt.show();